# Linear Regression Optimization

In this project, I implement and compare different optimization methods for multivariable linear regression using the California Housing dataset.

The methods used are:
- Batch Gradient Descent
- Stochastic Gradient Descent
- Mini-Batch Gradient Descent
- Newton's Method


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
import pandas as pd
import time


In [ ]:
housing = fetch_california_housing()

X = housing.data
y = housing.target

df = pd.DataFrame(X, columns=housing.feature_names)

df["Price"] = y
X = df[['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']].values
y = df['Price'].values

print(df.head())

# Standardize the features
mu = np.mean(X, axis=0)
sigma = np.std(X, axis=0)

X = (X - mu) / sigma

# Add a column of 1s for the intercept term
X = np.column_stack((np.ones(len(X)), X))

m = X.shape[0]


In [ ]:
def cost_fn(x, y, th):
    error = x @ th - y
    j = (error.T @ error) / 2
    return j


In [ ]:
def batch_grad(x, y, m, th, alpha, epochs):
    cost = np.zeros(epochs//50)
    for i in range(epochs):
        if ((i+1)%50 == 0):
            cost[(i+1)//50 - 1] = cost_fn(x, y, th)

        pred = x @ th
        err = pred - y
        gradient = (x.T @ err)/m
        th = th - alpha*gradient

    return th, cost


In [ ]:
thet1 = np.zeros(len(X[0]))
epochs_bgd = 4000

start = time.time()
thet1, cost_gd = batch_grad(X, y, m, thet1, 0.01, epochs_bgd)
end = time.time()

print(f"Time = {end - start}")
print("Parameter Array = ", thet1)

y_new1 = X @ thet1

ss_res = np.sum((y - y_new1)**2)
ss_tot = np.sum((y - np.mean(y))**2)

r2 = 1 - ss_res/ss_tot
print("R^2 score = ", r2)

ep_bgd = np.zeros(epochs_bgd//50)
for i in range(epochs_bgd//50):
    ep_bgd[i] = 50*(i+1)


Batch Gradient Descent produced a smooth convergence curve. Since the whole dataset is used for every update, the updates are stable, but it takes more iterations to reach the minimum.


In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y, y_new1,
            alpha=0.3,
            s=10)

plt.plot(
    [min(y), max(y)],
    [min(y), max(y)],
    c='red',
    linewidth=2
)

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")

plt.title("Linear Regression Predictions")

plt.grid(True)

plt.show()

In [ ]:
plt.plot(ep_bgd, cost_gd, label="BGD")

plt.legend()

plt.xlabel("Epochs")

plt.ylabel("Cost")

plt.title("Cost vs Epochs")

plt.show()

In [ ]:
def stochastic_gd(x, y, m, epochs, alpha, t):

    cost = np.zeros(epochs//50)
    for i in range(epochs):
        if ((i+1)%50 == 0):
            cost[(i+1)//50 - 1] = cost_fn(x, y, t)

        indices = np.random.permutation(m)
        for j in indices:
            pred = t.T @ x[j]
            err = pred - y[j]
            gradient = err*x[j]
            t = t - alpha*gradient

    return t, cost


In [ ]:
np.random.seed(42)

thet1 = np.zeros(len(X[0]))
epochs = 600

start = time.time()
thet1, cost_sgd = stochastic_gd(X, y, m, epochs, 0.00001, thet1)
end = time.time()

print(f"Time = {end - start}")
print("Parameter Array = ", thet1)

y_new2 = X @ thet1

ss_res = np.sum((y - y_new2)**2)
ss_tot = np.sum((y - np.mean(y))**2)

r2 = 1 - ss_res/ss_tot
print("R^2 score = ", r2)

ep = np.zeros(epochs//50)
for i in range(epochs//50):
    ep[i] = 50*(i+1)


Stochastic Gradient Descent showed faster initial learning than Batch Gradient Descent. Because it updates the parameters one example at a time, its convergence curve is more noisy. The random shuffling makes the updates less predictable, but it is closer to how stochastic optimization works.


In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y, y_new2,
            alpha=0.3,
            s=10)

plt.plot(
    [min(y), max(y)],
    [min(y), max(y)],
    c='red',
    linewidth=2
)

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")

plt.title("Linear Regression Predictions")

plt.grid(True)

plt.show()

In [ ]:
plt.plot(ep, cost_sgd, label="SGD")
plt.legend()

plt.xlabel("Epochs")

plt.ylabel("Cost")

plt.title("Optimization Comparison")

plt.show()

In [ ]:
# Newton's Method

def newton_method(x, y):
    # For linear regression, this gives the closed-form solution.
    theta = np.linalg.pinv(x.T @ x) @ x.T @ y
    return theta


In [ ]:
start = time.time()
th1 = newton_method(X, y)
end = time.time()

print(f"Time = {end - start}")
print("Parameter Array = ", th1)

y_new3 = X @ th1

ss_res = np.sum((y - y_new3)**2)
ss_tot = np.sum((y - np.mean(y))**2)

r2 = 1 - ss_res/ss_tot
print("R^2 score = ", r2)


The method gives the parameters almost immediately because it calculates the solution directly using matrix operations instead of updating the parameters for many epochs. This is very fast for this dataset, although matrix operations can become expensive when the number of features becomes very large.


In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y, y_new3,
            alpha=0.3,
            s=10)

plt.plot(
    [min(y), max(y)],
    [min(y), max(y)],
    c='red',
    linewidth=2
)

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")

plt.title("Linear Regression Predictions")

plt.grid(True)

plt.show()

In [ ]:
def mini_bgd(x, y, t, alpha, epochs, batch_size):
    cost = []
    for i in range(epochs):
        indices = np.random.permutation(m)
        x_shuff = X[indices]
        y_shuff = y[indices]
        for k in range(0, m, batch_size):
            if (k == (m//batch_size)*batch_size):
                x_new = x_shuff[k:]
                y_new = y_shuff[k:]
            else:
                x_new = x_shuff[k:k+batch_size]
                y_new = y_shuff[k:k+batch_size]
            pred = x_new @ t
            err = pred - y_new
            grad = (x_new.T)@err/batch_size
            t = t - alpha*grad
        if (i+1) % 50 == 0:
            cost.append(cost_fn(x_shuff, y_shuff, t))
    return t, cost



In [ ]:
def mini_bgd(x, y, t, alpha, epochs, batch_size):
    cost = []

    for i in range(epochs):
        indices = np.random.permutation(len(x))
        x_shuff = x[indices]
        y_shuff = y[indices]

        for k in range(0, len(x), batch_size):
            x_new = x_shuff[k:k+batch_size]
            y_new = y_shuff[k:k+batch_size]

            pred = x_new @ t
            err = pred - y_new
            grad = (x_new.T @ err) / len(x_new)
            t = t - alpha * grad

        if (i+1) % 50 == 0:
            cost.append(cost_fn(x_shuff, y_shuff, t))

    return t, cost


np.random.seed(42)

th4 = np.zeros(len(X[0]))
ep_minibgd = 300
batch_len = 64

start = time.time()
th4, cost_minibgd = mini_bgd(X, y, th4, 0.001, ep_minibgd, batch_len)
end = time.time()

print(f"Time = {end - start}")
y_new4 = X @ th4
print("Parameter Array = ", th4)

ss_res = np.sum((y - y_new4)**2)
ss_tot = np.sum((y - np.mean(y))**2)

r2 = 1 - ss_res/ss_tot
print("R^2 score = ", r2)

ep_mbgd = np.zeros(ep_minibgd//50)
for i in range(ep_minibgd//50):
    ep_mbgd[i] = 50*(i+1)


In [ ]:
Mini-Batch Gradient Descent gave a balance between Batch Gradient Descent and Stochastic Gradient Descent. It uses a small group of examples for each update, so it is less noisy than SGD while still making more frequent updates than BGD.


In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(y, y_new4,
            alpha=0.3,
            s=10)

plt.plot(
    [min(y), max(y)],
    [min(y), max(y)],
    c='red',
    linewidth=2
)

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")

plt.title("Mini-Batch GD Predictions")

plt.grid(True)

plt.show()


In [ ]:
plt.plot(ep_mbgd, cost_minibgd, label="Mini Batch GD")
plt.legend()

plt.xlabel("Epochs")
plt.ylabel("Cost")

plt.title("Mini-Batch GD Cost")

plt.show()


plt.plot(ep_bgd[:13], cost_gd[:13], label="BGD")
plt.plot(ep, cost_sgd, label="SGD")
plt.plot(ep_mbgd, cost_minibgd, label="Mini Batch GD")

plt.ylim(5000, 7500)
plt.xlim(0, 600)

plt.xlabel("Epochs")
plt.ylabel("Cost")
plt.title("Optimization Algorithm Comparison")

plt.legend()
plt.show()


The comparison graph shows the difference between the optimization methods.

- BGD produced the smoothest convergence but required more iterations.
- SGD converged faster but had more fluctuations.
- Mini-Batch GD gave a balance between stability and speed.
- The closed-form solution was very fast for this dataset.

Overall, the project helped me understand how different optimization methods behave when training a linear regression model.


In [ ]:
## Conclusion

This project implements and compares Batch Gradient Descent, Stochastic Gradient Descent, Mini-Batch Gradient Descent, and Newton's Method for multivariable linear regression on the California Housing dataset.

The main things compared were convergence, cost, running time, and R² score.
